In [1]:
from snn import Network, NodeGroup, EdgeGroup, NodePort, NodeList
from snn.model import IN, LIF, PSP

## Basic Syntax

In [2]:
net = Network()                       # instantiate network
net.layer = [NodeGroup(LIF(), 32),    # add a list of node groups
             NodeGroup(LIF(), 10)]  
net.dense = EdgeGroup(net.layer[0],   # add an edge group connecting
                      net.layer[1],   # the node groups
                      PSP())
net.inp = NodePort()                  # add an input port (not linked)
net.out = NodeList(net.layer[1][::2]) # add an output list (slicing)

net.build()       # build network topology
print(net)        # print the topological components

info: flattening network topology
(node) layer[0]
(node) layer[1]
(edge) dense: (node) layer[0] -> (node) layer[1]
(port) inp <- (no link)
(list) out


## Networks

In [3]:
import itertools

class Linear(Network):
    def __init__(self, size=256):
        super().__init__()      # base class initialization (required)
        self.size = size        # user-defined network parameters
        self.inp = NodePort()   # unsized node port (generates dependency)
        self.ctrl = NodePort(1) # sized node port (no dependency)

    def build(self):
        # Layers (using supplied parameters)
        self.layer = NodeGroup(LIF(), self.size)

        # Procedurally generating edges (using resolved port information)
        edges = [(i,j) for i,j in itertools.product(
            range(self.inp.size),range(self.layer.size))]

        # Connections (using node port placeholder)
        self.dense = EdgeGroup(self.inp, self.layer, PSP(), edges=edges)

        return

In [4]:
net = Network()                 # instantiate network
net.inp = NodeGroup(IN(), 12)   # add an input node group
net.ff = [Linear(32),           # add a list of networks
          Linear(10)]
# connect topological components (using dot notation)
net.connect(net.inp, net.ff[0].inp)
net.connect(net.ff[0].layer, net.ff[1].inp)
net.build()       # build network topology
print(net)        # print the topological components

info: adding list of networks ff
info: linking port ff[0].inp
info: waiting on source ff[0].layer
info: resolving dependency for ff[0].inp
info: building network ff[0]
info: linking port ff[1].inp
info: resolving dependency for ff[1].inp
info: building network ff[1]
info: flattening network topology
(node) inp
(port) ff[0].inp <- (node) inp
(port) ff[0].ctrl <- (no link)
(node) ff[0].layer
(edge) ff[0].dense: (port) ff[0].inp <- (node) inp -> (node) ff[0].layer
(port) ff[1].inp <- (node) ff[0].layer
(port) ff[1].ctrl <- (no link)
(node) ff[1].layer
(edge) ff[1].dense: (port) ff[1].inp <- (node) ff[0].layer -> (node) ff[1].layer


## Nodes and Edges

In [5]:
from dataclasses import dataclass
from snn.model import Neuron

@dataclass
class LIF(Neuron):
    model: str = 'LIF'       # model name
    voltage: float = 0.0     # individual parameter
    threshold: float = 1.0
    reset: float = 0.0,      # shared parameter
    leak: float = 1.0

res = NodeGroup(LIF(threshold=0.9),    # model parameter default
                size=3,                # node group size
                voltage=0.6,           # bulk parameter assignment
                leak=[0.5, 0.4, 0.3])  # individual parameter assignment
res.reset = 0.1                        # shared parameter assignment
res[1].voltage = 0.8                   # individual node parameter assignment

print(res.model)        # ['LIF']
print(res.voltage)      # [0.6 0.8 0.6]
print(res.threshold)    # [0.9 0.9 0.9]
print(res.reset)        # [(0.1,)]
print(res.leak)         # [0.5 0.4 0.3]

['LIF']
[0.6 0.8 0.6]
[0.9 0.9 0.9]
[(0.1,)]
[0.5 0.4 0.3]
